In [1]:
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import classification_report, confusion_matrix

import matplotlib.pyplot as plt
import seaborn as sns


In [2]:
RANGES_PATH = "C:\\Users\\mnabielizzuddin.radz\\Downloads\\EG556N_Asssement\\data\\NeuroFuzzy_EOR_Extracted_Tables.xlsx"
table1 = pd.read_excel(RANGES_PATH, sheet_name="Table1_Ranges", engine="openpyxl")

table1.head()

,EOR technique,Formation type,# projects,Depth min (ft),Depth max (ft),Porosity min (%),Porosity max (%),Permeability min (mD),Permeability max (mD),Oil gravity min (°API),Oil gravity max (°API),Oil viscosity min (cp),Oil viscosity max (cp),So at start min (%),So at start max (%),EOR production min (B/D),EOR production max (B/D)
0,Steam,Sandstone,113.0,250.0,5750.0,15.0,39.0,100.0,10000.0,8.0,22.0,18.00,500000.0,20.0,90.0,62.0,86000.0
1,Steam,Unconsolidated sands,26.0,175.0,3150.0,25.0,40.0,300.0,15000.0,9.0,25.0,175.00,200000.0,48.0,90.0,500.0,190000.0
2,Steam,Carbonates,6.0,550.0,1500.0,20.0,65.0,1.0,2000.0,10.0,29.0,26.00,4000.0,45.0,85.0,25.0,1200.0
3,Miscible CO2,Sandstone,50.0,1600.0,11950.0,10.0,28.0,9.0,2300.0,27.0,45.0,0.30,3.0,26.0,77.0,205.0,15000.0
4,Miscible CO2,Carbonates,83.0,4000.0,11100.0,4.0,24.0,0.1,5000.0,28.0,45.0,0.32,6.0,30.0,89.0,25.0,28300.0


In [3]:
def norm_tech(x):
    if pd.isna(x): 
        return np.nan
    x = str(x).strip()
    x = x.replace("CO22", "CO2").replace("*","")
    return x

def norm_form(x):
    if pd.isna(x):
        return np.nan
    x = str(x).strip().lower()
    if "sandstone" in x:
        return "Sandstone"
    if "unconsolidated" in x:
        return "Unconsolidated sands"
    if "carbonate" in x:
        return "Carbonates"
    return np.nan

table1["technique"] = table1["EOR technique"].apply(norm_tech)
table1["formation_category"] = table1["Formation type"].apply(norm_form)

table1 = table1.dropna(subset=["technique","formation_category"]).copy()
table1[["technique","formation_category"]].drop_duplicates().head(20)


,technique,formation_category
0,Steam,Sandstone
1,Steam,Unconsolidated sands
2,Steam,Carbonates
3,Miscible CO2,Sandstone
4,Miscible CO2,Carbonates
5,Miscible HC,Sandstone
6,Miscible HC,Carbonates
7,Polymer,Sandstone
8,Combustion,Sandstone
9,Combustion,Unconsolidated sands


In [4]:
ENV = {}

for _, r in table1.iterrows():
    key = (r["technique"], r["formation_category"])
    ENV[key] = {
        "depth": (r["Depth min (ft)"], r["Depth max (ft)"]),
        "por":   (r["Porosity min (%)"], r["Porosity max (%)"]),
        "perm":  (r["Permeability min (mD)"], r["Permeability max (mD)"]),
        "api":   (r["Oil gravity min (°API)"], r["Oil gravity max (°API)"]),
        "visc":  (r["Oil viscosity min (cp)"], r["Oil viscosity max (cp)"]),
        "so":    (r["So at start min (%)"], r["So at start max (%)"])
    }

len(ENV), list(ENV.keys())[:10]

(19,
 [('Steam', 'Sandstone'),
  ('Steam', 'Unconsolidated sands'),
  ('Steam', 'Carbonates'),
  ('Miscible CO2', 'Sandstone'),
  ('Miscible CO2', 'Carbonates'),
  ('Miscible HC', 'Sandstone'),
  ('Miscible HC', 'Carbonates'),
  ('Polymer', 'Sandstone'),
  ('Combustion', 'Sandstone'),
  ('Combustion', 'Unconsolidated sands')])

In [5]:
def trap_membership(x, L, U, alpha=0.2):
    # handles missing
    if pd.isna(x) or pd.isna(L) or pd.isna(U):
        return np.nan
    if U == L:
        return 1.0 if x == L else 0.0
    
    w = U - L
    a = L - alpha*w
    d = U + alpha*w
    
    # trapezoid points: (a, L, U, d)
    if x <= a or x >= d:
        return 0.0
    elif L <= x <= U:
        return 1.0
    elif a < x < L:
        return (x - a) / (L - a)
    else:  # U < x < d
        return (d - x) / (d - U)


In [6]:
df = table1.copy()

required = ["formation_category_raw"] + ["depth_ft_mid","porosity_pct_mid","perm_md_mid","api_mid","visc_cp_mid","so_pct_mid"]
missing = [col for col in required if col not in df.columns]

print("missing columns:", missing)

missing columns: ['formation_category_raw', 'depth_ft_mid', 'porosity_pct_mid', 'perm_md_mid', 'api_mid', 'visc_cp_mid', 'so_pct_mid']


In [7]:
df["formation_category_raw"] = df["formation_category"]

df["depth_ft_mid"] = df[["Depth min (ft)", "Depth max (ft)"]].mean(axis=1)
df["porosity_pct_mid"] = df[["Porosity min (%)", "Porosity max (%)"]].mean(axis=1)
df["perm_md_mid"] = df[["Permeability min (mD)", "Permeability max (mD)"]].mean(axis=1)
df["api_mid"] = df[["Oil gravity min (°API)", "Oil gravity max (°API)"]].mean(axis=1)
df["visc_cp_mid"] = df[["Oil viscosity min (cp)", "Oil viscosity max (cp)"]].mean(axis=1)
df["so_pct_mid"] = df[["So at start min (%)", "So at start max (%)"]].mean(axis=1)

TECHS_ALL = sorted(set(k[0] for k in ENV.keys()))

def score_for_tech(row, tech):
    form = row["formation_category_raw"]
    key = (tech, form)
    if key not in ENV:
        return np.nan

    e = ENV[key]
    m_depth = trap_membership(row["depth_ft_mid"], *e["depth"])
    m_por   = trap_membership(row["porosity_pct_mid"], *e["por"])
    m_perm  = trap_membership(row["perm_md_mid"], *e["perm"])
    m_api   = trap_membership(row["api_mid"], *e["api"])
    m_visc  = trap_membership(row["visc_cp_mid"], *e["visc"])
    m_so    = trap_membership(row["so_pct_mid"], *e["so"])
    return np.nanmin([m_depth, m_por, m_perm, m_api, m_visc, m_so])

for tech in TECHS_ALL:
    df[f"fuzzy_{tech}"] = df.apply(lambda r: score_for_tech(r, tech), axis=1)

df[[c for c in df.columns if c.startswith("fuzzy_")]].head()

C:\Users\mnabielizzuddin.radz\AppData\Local\Temp\ipykernel_29716\2766649931.py:25: RuntimeWarning: All-NaN axis encountered
  return np.nanmin([m_depth, m_por, m_perm, m_api, m_visc, m_so])
C:\Users\mnabielizzuddin.radz\AppData\Local\Temp\ipykernel_29716\2766649931.py:25: RuntimeWarning: All-NaN axis encountered
  return np.nanmin([m_depth, m_por, m_perm, m_api, m_visc, m_so])
C:\Users\mnabielizzuddin.radz\AppData\Local\Temp\ipykernel_29716\2766649931.py:25: RuntimeWarning: All-NaN axis encountered
  return np.nanmin([m_depth, m_por, m_perm, m_api, m_visc, m_so])


,fuzzy_Combustion,fuzzy_Hot water,fuzzy_Microbial,fuzzy_Miscible CO2,fuzzy_Miscible HC,fuzzy_Miscible acid gas,fuzzy_Nitrates,fuzzy_Polymer,fuzzy_Steam,fuzzy_Surfactants
0,0.0,0.0,0.0,0.0,0.0,NaN,0.0,0.0,1.0,0.0
1,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1.0,NaN
2,0.0,NaN,NaN,0.0,0.0,0.0,0.0,NaN,1.0,0.0
3,0.0,0.0,0.0,1.0,1.0,NaN,0.0,0.0,0.0,0.0
4,0.0,NaN,NaN,1.0,0.0,0.0,0.0,NaN,0.0,0.0


In [8]:
df_projects = df.copy()   # project-level extracted rows
df_ranges = table1.copy() # Table1_Ranges envelope table

In [9]:
# 1) Base numeric features (from your earlier notebook)
core_mid_cols = [
    "depth_ft_mid",
    "porosity_pct_mid",
    "perm_md_mid",
    "api_mid",
    "visc_cp_mid",
    "so_pct_mid",
]

if "depth_ft_span" not in df.columns:
    df["depth_ft_span"] = df["Depth max (ft)"] - df["Depth min (ft)"]
if "porosity_pct_span" not in df.columns:
    df["porosity_pct_span"] = df["Porosity max (%)"] - df["Porosity min (%)"]
if "perm_md_span" not in df.columns:
    df["perm_md_span"] = df["Permeability max (mD)"] - df["Permeability min (mD)"]
if "api_span" not in df.columns:
    df["api_span"] = df["Oil gravity max (°API)"] - df["Oil gravity min (°API)"]
if "visc_cp_span" not in df.columns:
    df["visc_cp_span"] = df["Oil viscosity max (cp)"] - df["Oil viscosity min (cp)"]
if "so_pct_span" not in df.columns:
    df["so_pct_span"] = df["So at start max (%)"] - df["So at start min (%)"]

core_span_cols = [
    "depth_ft_span",
    "porosity_pct_span",
    "perm_md_span",
    "api_span",
    "visc_cp_span",
    "so_pct_span",
]

for col in ["perm_md_mid", "visc_cp_mid", "perm_md_span", "visc_cp_span"]:
    log_col = col.replace("perm_md", "log_perm").replace("visc_cp", "log_visc")
    if log_col not in df.columns:
        df[log_col] = np.log(df[col].where(df[col] > 0))

numeric_features = core_mid_cols + core_span_cols + [
    "log_perm_mid", "log_visc_mid",
    "log_perm_span", "log_visc_span"
]

# 2) Formation one-hots
formation_features = [c for c in df.columns if c.startswith("form_")]

# 3) Fuzzy features
fuzzy_features = [c for c in df.columns if c.startswith("fuzzy_")]

FEATURES_ALL = numeric_features + formation_features + fuzzy_features

print("Numeric:", len(numeric_features))
print("Formation:", len(formation_features))
print("Fuzzy:", len(fuzzy_features))
print("Total features:", len(FEATURES_ALL))


Numeric: 16
Formation: 0
Fuzzy: 10
Total features: 26


In [10]:
# Work only on trainable set for NN training
df_nn = df_trainable.copy()

NameError: name 'df_trainable' is not defined

In [11]:

# Fill NaN fuzzy scores with 0 (means: no envelope match / not defined)
df_nn[fuzzy_features] = df_nn[fuzzy_features].fillna(0.0)

X = df_nn[FEATURES_ALL].values
y_text = df_nn["technique"].values

from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
y = le.fit_transform(y_text)

class_names = le.classes_
num_classes = len(class_names)

print("NN classes:", class_names)
print("num_classes:", num_classes)

NameError: name 'df_nn' is not defined

In [43]:
from sklearn.model_selection import train_test_split

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=SEED, stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=SEED, stratify=y_temp
)

print("Train:", X_train.shape, "Val:", X_val.shape, "Test:", X_test.shape)


NameError: name 'SEED' is not defined